# Production Patterns for RAG

Notebooks prove RAG works on **c1–c5**; production requires **reliability**, **observability**, **security**, and **cost control** at scale. A RAG service must handle concurrent users, degrade gracefully when OpenAI or the vector DB is slow, and ship index updates without breaking answers.

This notebook covers production architecture, API contracts, versioned artifacts (index + prompt + model), caching, async/timeouts, rate limits, structured tracing, blue/green index deploys, security, cost levers, FastAPI integration patterns, ingest workers, health checks, and disaster recovery — building on **03. LLM API Integration** FastAPI apps and eval gates from **09**.

Prerequisites: **09. Evaluating RAG End-to-End**, **04. Knowledge Base and Ingest**, **08. RAG Failure Modes**, **03. LLM API Integration**. Next: **11. Conversational and Multi-Turn RAG**.

In [ ]:
import hashlib
import json
import time
import uuid
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any

import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"
PROMPT_VERSION = "rag_support_v3"
INDEX_VERSION = "notes_api_2026-07-09_v1"

print("Config loaded — index:", INDEX_VERSION)

---

## 1. Notebook vs Production

```
Notebook                         Production
────────                         ──────────
Re-run cells manually              Scheduled ingest + API service
In-memory Chroma                   Persistent pgvector / Pinecone
print(retrieved_ids)               Structured logs + trace_id
Hard-coded API key in cell         Vault / env secrets
Eyeball one answer                 Recall@k CI gate (**09**)
```

| Concern | Notebook habit | Production pattern |
|---------|----------------|-------------------|
| **Reproducibility** | Ad hoc re-embed | Versioned index + manifest |
| **Failures** | Retry cell | Timeouts, circuit breakers, fallbacks |
| **Scale** | 5 chunks | Millions — batch ingest (**04**) |
| **Ownership** | One author | Runbooks, on-call, SLIs |

---

## 2. Production Architecture

Separate **read path** (low-latency queries) from **write path** (batch ingest).

```
Client ──► API Gateway ──► RAG Service (FastAPI)
                              │
                    ┌─────────┼─────────┐
                    ▼         ▼         ▼
                 Redis     Vector DB   OpenAI
                 cache     (Chroma/     APIs
                           pgvector)
                    ▲
              Ingest worker (cron / queue)
```

| Component | Responsibility |
|-----------|----------------|
| **RAG API** | `/ask`, retrieval, prompt, generate |
| **Vector DB** | Durable index, metadata filters |
| **Cache** | Embed + retrieval + optional answer cache |
| **Ingest worker** | Offline pipeline (**04**) |
| **Eval job** | Nightly regression (**09**) |

---

## 3. Version Everything

Every answer must trace to the **exact artifacts** that produced it.

| Artifact | Version as | Example |
|----------|------------|--------|
| Embedding model | config | `text-embedding-3-small` |
| Chunk strategy | slug | `chunk_v2_512_20overlap` |
| Index / collection | name suffix | `notes_prod_v3` |
| Prompt template | git tag | `rag_support_v3` |
| Chat model | config | `gpt-4o-mini` |
| Build id | timestamp | `2026-07-09T10:00:00Z` |

**Golden rule:** bump cache keys and run eval (**09**) when **any** of these change.

In [ ]:
@dataclass
class RAGManifest:
    index_version: str
    embedding_model: str
    chunk_strategy: str
    prompt_version: str
    chat_model: str
    build_id: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    def cache_namespace(self) -> str:
        return f"{self.index_version}|{self.prompt_version}|{self.embedding_model}"


manifest = RAGManifest(
    index_version=INDEX_VERSION,
    embedding_model=EMBED_MODEL,
    chunk_strategy="markdown_512_20overlap",
    prompt_version=PROMPT_VERSION,
    chat_model=CHAT_MODEL,
)
print(json.dumps(asdict(manifest), indent=2))

---

## 4. API Contract

| Endpoint | Purpose |
|----------|--------|
| `POST /ask` | Question → answer + sources + trace_id |
| `POST /retrieve` | Debug retrieval only (no LLM) |
| `GET /health` | Liveness + index version |
| `POST /feedback` | Thumb up/down linked to trace_id |

### 4.1 Response Shape

```json
{
  "answer": "Run alembic upgrade head...",
  "sources": [
    {"id": "c2", "snippet": "...", "path": "docs/migrations.md"}
  ],
  "trace_id": "550e8400-e29b-41d4-a716-446655440000",
  "index_version": "notes_api_2026-07-09_v1",
  "prompt_version": "rag_support_v3"
}
```

Clients and support tools use `trace_id` to pull logs for incident review.

In [ ]:
@dataclass
class AskResponse:
    answer: str
    sources: list[dict]
    trace_id: str
    index_version: str
    prompt_version: str


def build_response(answer: str, chunk_ids: list[str], manifest: RAGManifest) -> dict:
    return asdict(AskResponse(
        answer=answer,
        sources=[{"id": cid, "snippet": f"[{cid}] ..."} for cid in chunk_ids],
        trace_id=str(uuid.uuid4()),
        index_version=manifest.index_version,
        prompt_version=manifest.prompt_version,
    ))


print(json.dumps(build_response("Run alembic upgrade head.", ["c2"], manifest), indent=2))

---

## 5. Modular Service Layout

Mirror notebook **02** / **03** modules — injectable for tests:

```
app/
  main.py           # FastAPI routes
  config.py         # env + manifest
  services/
    retriever.py    # search(query, k) -> chunks
    prompt.py       # build_messages(question, chunks)
    generator.py    # complete(messages) -> str
    orchestrator.py # wire + trace logging
  workers/
    ingest.py       # offline job (**04**)
```

Unit-test `retriever` and `prompt` without calling OpenAI.

---

## 6. Secrets and Configuration

| Secret | Storage | Never |
|--------|---------|-------|
| `OPENAI_API_KEY` | Env / vault | Commit to git |
| Vector DB credentials | Same | Log in plaintext |
| User API keys (your RAG API) | Hashed DB | Return in responses |

Notebooks use `sk-proj-placeholder-replace-with-your-real-key`. Production uses `.env` locally and managed secrets (AWS SM, Azure KV) in deploy.

Load config once at startup — not per request.

---

## 7. Caching Strategy

| Cache layer | Key includes | TTL | Invalidate on |
|-------------|--------------|-----|---------------|
| Query embedding | `hash(question)` | Hours | Embed model change |
| Retrieval results | `index_version + embed_key` | Minutes | Index bump |
| Full answer | `index + prompt + question` | Short / off | Prompt or index change |

Always include **`index_version`** in cache keys — otherwise users get answers from a stale KB after ingest.

In [ ]:
def normalize_question(q: str) -> str:
    return " ".join(q.strip().lower().split())


def cache_key(question: str, manifest: RAGManifest, layer: str = "answer") -> str:
    raw = f"{layer}|{manifest.cache_namespace()}|{normalize_question(question)}"
    return hashlib.sha256(raw.encode()).hexdigest()[:24]


q = "How does JWT authentication work?"
print("embed key:     ", cache_key(q, manifest, "embed"))
print("retrieval key: ", cache_key(q, manifest, "retrieve"))
print("answer key:    ", cache_key(q, manifest, "answer"))

---

## 8. Async, Timeouts, and Graceful Degradation

| Stage | Typical timeout | On failure |
|-------|-----------------|------------|
| Embed query | 5s | Retry once; else 503 |
| Vector search | 2s | Retry; fallback read replica |
| LLM completion | 30–60s | Stream partial; or retrieval-only snippet |

Use async FastAPI + async OpenAI client for concurrent users. **Never** block the event loop on sync embed calls at scale.

```python
@app.post("/ask")
async def ask(body: AskRequest):
    ...
```

---

## 9. Rate Limiting and Quotas

| Control | Protects |
|---------|----------|
| Per-user RPM on `/ask` | LLM spend abuse |
| Per-tenant embed quota | Ingest cost |
| `max_tokens` cap | Runaway completions |
| Max `k` per tier | Context cost (**06**) |

Queue **ingest** jobs separately from **query** traffic — batch embed should not starve live users.

---

## 10. Observability — RAG Trace

Structured log per request (JSON) — links to **08** `RAGTrace` and **09** eval.

| Field | Purpose |
|-------|--------|
| `trace_id` | Support correlation |
| `latency_embed_ms` | Bottleneck detection |
| `latency_search_ms` | Vector DB health |
| `latency_llm_ms` | Model latency |
| `retrieved_ids` | Retrieval audit |
| `prompt_tokens`, `completion_tokens` | Cost |
| `index_version`, `prompt_version` | Reproduce answer |

In [ ]:
@dataclass
class RAGTraceLog:
    trace_id: str
    question: str
    retrieved_ids: list[str]
    latency_embed_ms: float
    latency_search_ms: float
    latency_llm_ms: float
    prompt_tokens: int
    completion_tokens: int
    index_version: str
    prompt_version: str
    chat_model: str
    status: str = "ok"

    def to_json(self) -> str:
        return json.dumps(asdict(self))

    @property
    def total_latency_ms(self) -> float:
        return self.latency_embed_ms + self.latency_search_ms + self.latency_llm_ms


trace = RAGTraceLog(
    trace_id=str(uuid.uuid4()),
    question="How do clients authenticate?",
    retrieved_ids=["c3"],
    latency_embed_ms=45,
    latency_search_ms=12,
    latency_llm_ms=890,
    prompt_tokens=620,
    completion_tokens=85,
    index_version=INDEX_VERSION,
    prompt_version=PROMPT_VERSION,
    chat_model=CHAT_MODEL,
)
print(trace.to_json())
print(f"total_latency_ms={trace.total_latency_ms}")

In [ ]:
latencies = {
    "embed": trace.latency_embed_ms,
    "search": trace.latency_search_ms,
    "llm": trace.latency_llm_ms,
}
plt.figure(figsize=(6, 3.5))
plt.bar(latencies.keys(), latencies.values(), color=["#4c72b0", "#55a868", "#c44e52"])
plt.ylabel("ms (example trace)")
plt.title("Where time goes in one RAG request")
plt.tight_layout()
plt.show()

LLM generation usually dominates latency — but embed/search spikes indicate infrastructure issues.

---

## 11. Blue/Green Index Deploys

```
Build collection v2 ──► run Recall@k eval (**09**) ──► flip router pointer ──► retire v1
```

| Step | Action |
|------|--------|
| 1 | Ingest to **new** collection name |
| 2 | Run eval against v2; compare to v1 baseline |
| 3 | Update `INDEX_VERSION` in API config (atomic) |
| 4 | Bust retrieval/answer caches |
| 5 | Keep v1 for rollback window (7–14 days) |

Never serve half-updated indexes during long reindex — users get inconsistent answers.

In [ ]:
def deploy_index(new_version: str, eval_recall: float, min_recall: float = 0.85) -> str:
    if eval_recall < min_recall:
        raise ValueError(f"Recall {eval_recall:.2f} below gate {min_recall}")
    return new_version  # atomic config swap in real deploy


try:
    active = deploy_index("notes_api_2026-07-10_v2", eval_recall=0.91)
    print("Deployed:", active)
except ValueError as e:
    print("Rollback — eval failed:", e)

---

## 12. Release Workflow with Eval Gates

```
PR merge ──► ingest staging index ──► pytest rag_eval ──► deploy staging ──► smoke ──► prod
```

| Gate | Threshold (example) |
|------|---------------------|
| Recall@3 | ≥ 0.85 |
| Refusal accuracy | ≥ 0.90 |
| Citation valid rate | ≥ 0.95 |
| p95 latency | < 3s |

Block release if any gate fails vs `main` baseline (**09**).

---

## 13. Security Checklist

| Item | Action | Reference |
|------|--------|----------|
| Prompt injection | System guardrails + ingest scan | **05**, **08** |
| ACL metadata | Query-time `where` filter | **04**, **07** |
| PII in logs | Redact before storage | **04** |
| Auth on RAG API | JWT / API keys | **03** FastAPI |
| Secret rotation | Rotate OpenAI key on leak | Ops runbook |
| Tenant isolation | Separate collections or filter | **04**, **05.10** |

---

## 14. Cost Management

| Lever | Savings | Trade-off |
|-------|---------|----------|
| Smaller chat model (`gpt-4o-mini`) | High | Quality on hard queries |
| Lower k after MMR (**07**) | Medium | Recall risk |
| Cache frequent questions | High | Stale after index bump |
| Batch nightly embed | Medium | Freshness lag |
| Shorter system prompt | Low | Less guidance |
| Streaming responses | UX only | Same token cost |

Log `prompt_tokens + completion_tokens` per `trace_id` — allocate cost per team/tenant.

In [ ]:
def estimate_cost_usd(prompt_tokens: int, completion_tokens: int,
                      price_in: float = 0.15, price_out: float = 0.60) -> float:
    """Illustrative $/1M tokens for gpt-4o-mini-class pricing."""
    return (prompt_tokens * price_in + completion_tokens * price_out) / 1_000_000


cost = estimate_cost_usd(trace.prompt_tokens, trace.completion_tokens)
print(f"Example request cost: ${cost:.6f}")
print(f"At 10k requests/day: ${cost * 10_000:.2f}/day (illustrative)")

---

## 15. FastAPI Integration Sketch

Injectable services — same pattern as **03. LLM API Integration** apps.

In [ ]:
# Illustrative orchestrator (no server started)
class Retriever:
    def search(self, question: str, k: int = 5) -> list[dict]:
        return [{"id": "c3", "text": "JWT bearer tokens..."}]


class Generator:
    def answer(self, question: str, chunks: list[dict]) -> str:
        return f"Answer about {question} using {chunks[0]['id']}"


class RAGOrchestrator:
    def __init__(self, retriever: Retriever, generator: Generator, manifest: RAGManifest):
        self.retriever = retriever
        self.generator = generator
        self.manifest = manifest

    def ask(self, question: str) -> dict:
        t0 = time.perf_counter()
        trace_id = str(uuid.uuid4())
        chunks = self.retriever.search(question, k=5)
        t_search = (time.perf_counter() - t0) * 1000
        answer = self.generator.answer(question, chunks)
        return build_response(answer, [c["id"] for c in chunks], self.manifest)


orch = RAGOrchestrator(Retriever(), Generator(), manifest)
print(json.dumps(orch.ask("How do clients authenticate?"), indent=2))

---

## 16. Health Checks

| Check | `GET /health` includes |
|-------|------------------------|
| API process | `status: ok` |
| Vector DB | `collection_count` or ping |
| Index version | `index_version` string |
| Embed model | `embedding_model` |
| Optional smoke | Last eval Recall@3 score |

Liveness ≠ readiness — readiness fails if vector DB is unreachable even if FastAPI runs.

In [ ]:
def health_payload(manifest: RAGManifest, vector_ok: bool = True, chunk_count: int = 5) -> dict:
    return {
        "status": "ok" if vector_ok else "degraded",
        "index_version": manifest.index_version,
        "embedding_model": manifest.embedding_model,
        "prompt_version": manifest.prompt_version,
        "chunk_count": chunk_count,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }


print(json.dumps(health_payload(manifest), indent=2))

---

## 17. Disaster Recovery

| Scenario | Response |
|----------|----------|
| **OpenAI outage** | Queue requests; return retrieval snippets + banner |
| **Vector DB down** | Serve stale cache with `degraded` flag |
| **Bad deploy** (Recall drop) | Auto-rollback index pointer to v1 |
| **API key leak** | Rotate key; audit usage logs |
| **Poisoned ingest** | Halt worker; restore index from last good manifest |

Document runbooks before incidents — not during.

---

## 18. Multi-Tenant Considerations

| Pattern | Isolation | Ops |
|---------|-----------|-----|
| Collection per tenant | Strong | More collections |
| Shared index + `tenant_id` filter | Weaker | One index to manage |
| Separate embed indexes per region | Data residency | Compliance |

Per-tenant `INDEX_VERSION` and eval suites prevent one customer's bad ingest from affecting others.

---

## 19. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| No `trace_id` | Cannot debug prod failures | Log every request |
| Cache without index version | Stale answers after ingest | Namespace keys |
| Deploy without eval gate | Recall regression in prod | **09** in CI |
| Monolith RAG function | Untestable | Modular services |
| Sync embed at high QPS | Timeouts | Async + pool |
| Log full prompts with PII | Compliance breach | Redact / sample |

---

## 20. Summary

Production RAG is an **operated service**: versioned index + prompt + model, structured APIs with `trace_id`, caching keyed on `index_version`, async timeouts, observability per stage, blue/green deploys with **09** eval gates, security controls from **04–08**, and disaster runbooks.

Demonstrations covered `RAGManifest`, cache keys, `RAGTraceLog`, latency breakdown, deploy gating, cost estimation, orchestrator sketch, and health payloads.

Pair these patterns with eval regression on every release. Notebook **11** extends the read path for **multi-turn** chat.

Back: **09. Evaluating RAG End-to-End**. Next: **11. Conversational and Multi-Turn RAG**.